<a href="https://colab.research.google.com/github/eoinleen/Biophysics-general/blob/main/Dianthus_v18.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================================
# SETTINGS - edit these per experiment
# ============================================================================

# TOTAL target in the well (labelled + unlabelled), in nM. This is what the
# quadratic needs. It is NOT the dye concentration.
#   total = dye_in_well / labelled_fraction
#   e.g. K11 tracer: 10 nM dye / 0.64 = 15.6 nM;  K33: 10 / 0.56 = 17.7 nM
# One value per run, so analyse tracers with different totals in separate files.
TOTAL_TARGET_NM = 50
TRACER_DYE_NM   = 12.5      # reference only (reported, not used in the fit)

# Free-text descriptions used in the auto-generated methods paragraph.
TRACER_DESCRIPTION = ("K11-linked diubiquitin labelled on a single cysteine "
                      "with RED-MALEIMIDE 2nd Generation dye")
TITRANT_NAME = "binder"
X_LABEL = "[Binder] (nM)"

# Input layout (same as before): row 1 = sample names (one per column pair),
# row 2 = column headers, data below; columns alternate concentration (nM) / ratio.
# Column pairs that share a sample name are treated as repeats of that sample.
DATA_START_ROW = 2          # 0-based row index where numbers start

SHOW_RESIDUALS = True      # False = binding curves only (no residual panels)

N_BOOTSTRAP = 500
RANDOM_SEED = 1
OUTPUT_DIR = "dianthus_results"

In [ ]:

# ============================================================================
# Dianthus spectral shift analysis v18
# ----------------------------------------------------------------------------
# One pre-specified model for every dataset: 1:1 binding with ligand depletion
# (Morrison quadratic). Repeats of a sample are fitted globally with a shared
# Kd and their own baselines. Alternative models are run only as diagnostics
# and never replace the reported Kd.
# ============================================================================

import os, shutil, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

T = float(TOTAL_TARGET_NM)
rng = np.random.default_rng(RANDOM_SEED)

# ----------------------------------------------------------------------------
# Models (concentrations in nM throughout; Kd fitted as log10 for conditioning)
# ----------------------------------------------------------------------------

def frac_bound(c, kd, t=T):
    """Morrison quadratic, written in the cancellation-free form
    f = 2c / (b + sqrt(b^2 - 4ct)),  b = c + t + kd  (identical algebraically)."""
    c = np.asarray(c, float)
    b = c + t + kd
    return 2 * c / (b + np.sqrt(np.maximum(b * b - 4 * c * t, 0)))


def frac_bound_hill(c, kd, n, t=T):
    """Diagnostic only: Hill binding with depletion, mass balance solved by
    bisection for free ligand (L_tot = L_free + t * theta(L_free))."""
    c = np.asarray(c, float)
    lo, hi = np.zeros_like(c), c.copy()
    for _ in range(60):
        mid = 0.5 * (lo + hi)
        theta = 1 / (1 + (kd / np.maximum(mid, 1e-30)) ** n)
        over = mid + t * theta > c
        hi, lo = np.where(over, mid, hi), np.where(over, lo, mid)
    lf = 0.5 * (lo + hi)
    return 1 / (1 + (kd / np.maximum(lf, 1e-30)) ** n)


# Each model: shared params (after logKd) and per-repeat params.
MODELS = {
    "basic": dict(shared=[],    per=["R_unbound", "R_bound"]),
    "nonspecific": dict(shared=[], per=["R_unbound", "R_bound", "ns_slope"]),
    "hill": dict(shared=["n"],  per=["R_unbound", "R_bound"]),
}


def _predict(model, c, rep, p, n_rep):
    m = MODELS[model]
    kd = 10 ** p[0]
    ns, npr = len(m["shared"]), len(m["per"])
    per = p[1 + ns:].reshape(n_rep, npr)[rep]
    ru, rb = per[:, 0], per[:, 1]
    if model == "hill":
        f = frac_bound_hill(c, kd, p[1])
    else:
        f = frac_bound(c, kd)
    y = ru + f * (rb - ru)
    if model == "nonspecific":
        y = y + per[:, 2] * c
    return y


def global_fit(model, c, y, rep, n_rep, p0=None):
    """Fit all repeats of one sample at once. Returns params, residuals."""
    m = MODELS[model]
    cpos = c[c > 0]
    lo_kd, hi_kd = np.log10(cpos.min() / 100), np.log10(cpos.max() * 100)
    if p0 is None:
        # Baseline guesses from the lowest / highest concentrations of each repeat
        # (data are sorted, so order on the plate does not matter).
        per0 = []
        for r in range(n_rep):
            yr = y[rep == r]
            g = [np.mean(yr[:2]), np.mean(yr[-2:])]
            if model == "nonspecific":
                g.append(0.0)
            per0 += g
        shared0 = [1.0] if model == "hill" else []
        p0 = [np.log10(np.median(cpos))] + shared0 + per0
    lb = [lo_kd] + ([0.3] if model == "hill" else []) + [-np.inf] * (len(p0) - 1 - len(m["shared"]))
    ub = [hi_kd] + ([5.0] if model == "hill" else []) + [np.inf] * (len(p0) - 1 - len(m["shared"]))
    if model == "nonspecific":
        # Bound the slope so it cannot impersonate the binding transition:
        # at the top concentration it may contribute at most the full amplitude.
        amp = np.ptp(y)
        for r in range(n_rep):
            i = 1 + 3 * r + 2
            lb[i], ub[i] = -amp / cpos.max(), amp / cpos.max()
    p0 = np.clip(p0, np.array(lb) + 1e-9, np.array(ub) - 1e-9)
    f = lambda cc, *p: _predict(model, cc, rep, np.array(p), n_rep)
    p, _ = curve_fit(f, c, y, p0=p0, bounds=(lb, ub), maxfev=20000)
    return p, y - f(c, *p)


def aicc(rss, n, k):
    if n - k - 1 <= 0 or rss <= 0:
        return np.inf
    return n * np.log(rss / n) + 2 * k + 2 * k * (k + 1) / (n - k - 1)


def bootstrap_kd(c, y, rep, n_rep, p_hat, resid):
    """Residual bootstrap of the global basic fit. Residuals are resampled
    within each repeat, so each repeat keeps its own noise level."""
    fitted = y - resid
    kds = []
    for _ in range(N_BOOTSTRAP):
        yb = fitted.copy()
        for r in range(n_rep):
            idx = np.where(rep == r)[0]
            yb[idx] += rng.choice(resid[idx], idx.size, replace=True)
        try:
            pb, _ = global_fit("basic", c, yb, rep, n_rep, p0=p_hat)
            kds.append(10 ** pb[0])
        except Exception:
            pass
    kds = np.array(kds)
    if kds.size < 0.8 * N_BOOTSTRAP:
        return np.nan, np.nan, kds.size
    return *np.percentile(kds, [2.5, 97.5]), kds.size


# ----------------------------------------------------------------------------
# Loading
# ----------------------------------------------------------------------------

def load(path):
    raw = (pd.read_csv(path, header=None) if path.lower().endswith(".csv")
           else pd.read_excel(path, header=None))
    samples = {}
    for i in range(raw.shape[1] // 2):
        name = raw.iat[0, 2 * i]
        if pd.isna(name):
            name = raw.iat[0, 2 * i + 1]
        name = str(name).strip().strip('"\'') if pd.notna(name) else f"Sample {i+1}"
        c = pd.to_numeric(raw.iloc[DATA_START_ROW:, 2 * i], errors="coerce").to_numpy()
        y = pd.to_numeric(raw.iloc[DATA_START_ROW:, 2 * i + 1], errors="coerce").to_numpy()
        ok = np.isfinite(c) & np.isfinite(y) & (c >= 0)      # drop a row if EITHER is missing
        c, y = c[ok], y[ok]
        order = np.argsort(c)                                # plate order irrelevant
        samples.setdefault(name, []).append((c[order], y[order]))
    return samples


# ----------------------------------------------------------------------------
# Per-sample analysis
# ----------------------------------------------------------------------------

def analyse_sample(name, repeats):
    n_rep = len(repeats)
    c = np.concatenate([r[0] for r in repeats])
    y = np.concatenate([r[1] for r in repeats])
    rep = np.concatenate([np.full(len(r[0]), i) for i, r in enumerate(repeats)])

    out = dict(sample=name, n_repeats=n_rep, n_points=len(c), total_target_nM=T)

    p, resid = global_fit("basic", c, y, rep, n_rep)
    kd = 10 ** p[0]
    lo, hi, n_ok = bootstrap_kd(c, y, rep, n_rep, p, resid)
    out.update(Kd_nM=kd, Kd_CI95_low=lo, Kd_CI95_high=hi)

    # Each repeat on its own (same model): the spread is the honest error bar.
    kd_each = []
    for r in range(n_rep):
        try:
            pr, _ = global_fit("basic", c[rep == r], y[rep == r], np.zeros((rep == r).sum(), int), 1)
            kd_each.append(10 ** pr[0])
        except Exception:
            kd_each.append(np.nan)
    kd_each = np.array(kd_each)
    out["Kd_each_repeat_nM"] = "; ".join(f"{k:.3g}" for k in kd_each)
    if n_rep >= 2:
        out["Kd_repeats_mean_nM"] = np.nanmean(kd_each)
        out["Kd_repeats_SD_nM"] = np.nanstd(kd_each, ddof=1)

    # Quality numbers
    amp = np.abs(p[2::2] - p[1::2])                 # |R_bound - R_unbound| per repeat
    noise = np.std(resid, ddof=len(p))
    out["signal_to_noise"] = np.mean(amp) / noise if noise > 0 else np.inf
    out["frac_bound_at_top"] = float(frac_bound(c.max(), kd))

    # Diagnostics: do other models fit clearly better?  (never change the Kd)
    a_basic = aicc(np.sum(resid ** 2), len(c), len(p))
    for alt in ("nonspecific", "hill"):
        try:
            pa, ra = global_fit(alt, c, y, rep, n_rep)
            out[f"dAICc_{alt}"] = a_basic - aicc(np.sum(ra ** 2), len(c), len(pa))
            if alt == "hill":
                out["hill_n"] = pa[1]
        except Exception:
            out[f"dAICc_{alt}"] = np.nan

    flags = []
    if kd < T / 2:
        flags.append(f"Kd below resolution limit (~{T/2:.3g} nM): treat as upper bound")
    if out["frac_bound_at_top"] < 0.8:
        flags.append(f"not saturated ({100*out['frac_bound_at_top']:.0f}% bound at top conc.)")
    if c.max() < 3 * kd:
        flags.append("top concentration < 3x Kd")
    if out["signal_to_noise"] < 5:
        flags.append("low signal/noise")
    if out.get("dAICc_nonspecific", 0) > 6:
        flags.append("non-specific term strongly favoured: inspect")
    if out.get("dAICc_hill", 0) > 6:
        flags.append(f"Hill strongly favoured (n={out.get('hill_n', np.nan):.2f}): inspect")
    if n_rep >= 2 and np.nanmax(kd_each) / np.nanmin(kd_each) > 3:
        flags.append("repeats disagree >3-fold")
    if np.isnan(lo):
        flags.append("bootstrap unstable")
    out["flags"] = " | ".join(flags) if flags else "none"

    plot_data = dict(c=c, y=y, rep=rep, p=p, resid=resid, kd=kd, n_rep=n_rep)
    return out, plot_data


# ----------------------------------------------------------------------------
# Plotting: normalised fraction bound (each repeat scaled by its own baselines)
# with residuals underneath
# ----------------------------------------------------------------------------

MARKERS = "osD^vP*X"


def draw(ax, ax_res, name, d, res):
    c, y, rep, p, n_rep = d["c"], d["y"], d["rep"], d["p"], d["n_rep"]
    ru, rb = p[1::2][rep], p[2::2][rep]
    fobs = (y - ru) / (rb - ru)
    fres = d["resid"] / (rb - ru)
    pos = c > 0
    for r in range(n_rep):
        k = (rep == r) & pos
        ax.scatter(c[k], fobs[k], s=28, marker=MARKERS[r % len(MARKERS)],
                   facecolor="none", edgecolor="k", linewidth=0.9, label=f"repeat {r+1}")
        if ax_res is not None:
            ax_res.scatter(c[k], fres[k], s=12, marker=MARKERS[r % len(MARKERS)], color="k")
    xs = np.logspace(np.log10(c[pos].min()), np.log10(c.max()), 300)
    ax.plot(xs, frac_bound(xs, d["kd"]), color="#c0392b", lw=1.8)
    for a in (ax, ax_res):
        if a is not None:
            a.set_xscale("log")
            a.spines[["top", "right"]].set_visible(False)
    ax.set_ylim(-0.15, 1.15)
    ax.set_ylabel("Fraction bound")
    if ax_res is not None:
        ax_res.axhline(0, color="0.5", lw=0.8)
        ax_res.set_ylabel("Resid.")
        ax_res.set_xlabel(X_LABEL)
        ax.tick_params(labelbottom=False)
    else:
        ax.set_xlabel(X_LABEL)
    ci = (f" ({res['Kd_CI95_low']:.3g}–{res['Kd_CI95_high']:.3g})"
          if np.isfinite(res["Kd_CI95_low"]) else "")
    rel = "≤" if res["Kd_nM"] < T / 2 else "="
    ax.set_title(f"{name}\nK$_d$ {rel} {res['Kd_nM']:.3g} nM{ci}", fontsize=10)
    if n_rep > 1:
        ax.legend(frameon=False, fontsize=7, loc="upper left")


def plot_sample(name, d, res, path):
    if SHOW_RESIDUALS:
        fig, (ax, axr) = plt.subplots(2, 1, figsize=(4.2, 4.2), sharex=True,
                                      gridspec_kw=dict(height_ratios=[3, 1], hspace=0.08))
    else:
        fig, ax = plt.subplots(figsize=(4.2, 3.4))
        axr = None
    draw(ax, axr, name, d, res)
    fig.savefig(path + ".png", dpi=300, bbox_inches="tight")
    fig.savefig(path + ".svg", bbox_inches="tight")
    plt.close(fig)


def plot_grid(items, path, ncol=3):
    n = len(items)
    nrow = int(np.ceil(n / ncol))
    fig = plt.figure(figsize=(4.0 * ncol, (4.6 if SHOW_RESIDUALS else 3.8) * nrow))
    outer = fig.add_gridspec(nrow, ncol, hspace=0.45 if SHOW_RESIDUALS else 0.6, wspace=0.35)
    for i, (name, d, res) in enumerate(items):
        r, col = divmod(i, ncol)
        if SHOW_RESIDUALS:
            inner = outer[r, col].subgridspec(2, 1, height_ratios=[3, 1], hspace=0.08)
            ax = fig.add_subplot(inner[0])
            axr = fig.add_subplot(inner[1], sharex=ax)
        else:
            ax, axr = fig.add_subplot(outer[r, col]), None
        draw(ax, axr, name, d, res)
    for ext in ("png", "svg", "pdf"):
        fig.savefig(f"{path}.{ext}", dpi=300, bbox_inches="tight")
    plt.show() if "ipykernel" in sys.modules else None
    plt.close(fig)


# ----------------------------------------------------------------------------
# Methods text
# ----------------------------------------------------------------------------

def methods_text():
    return f"""Data analysis. For each sample, the 670/650 nm emission ratio (R) was fitted
as a function of total {TITRANT_NAME} concentration (C) to a 1:1 binding model
that accounts for ligand depletion (Morrison quadratic):

  R = R_unbound + (R_bound - R_unbound) * f,
  f = [(C + T + Kd) - sqrt((C + T + Kd)^2 - 4*C*T)] / (2*T)

where T is the total concentration of tracer ({TRACER_DESCRIPTION}),
labelled plus unlabelled ({T:g} nM in the well; {TRACER_DYE_NM:g} nM dye). Independent
repeats were fitted globally with a shared Kd and repeat-specific R_unbound and
R_bound, by non-linear least squares (SciPy curve_fit; Kd fitted on a log10 scale).
The 95% confidence interval for Kd was obtained by residual bootstrap ({N_BOOTSTRAP}
resamples, residuals resampled within each repeat; 2.5th-97.5th percentiles).
Each repeat was also fitted separately, and the mean +/- SD of these Kd values is
reported. Because the midpoint of the curve lies at Kd + T/2, Kd values below
T/2 ({T/2:.3g} nM) are reported as upper bounds. Models including a linear
non-specific term or a Hill coefficient were compared by AICc as diagnostics only.
"""


# ----------------------------------------------------------------------------
# Run
# ----------------------------------------------------------------------------

def run(path):
    if os.path.isdir(OUTPUT_DIR):
        shutil.rmtree(OUTPUT_DIR)
    os.makedirs(OUTPUT_DIR)
    samples = load(path)
    print(f"Total target in well: {T:g} nM  (dye {TRACER_DYE_NM:g} nM)  |  "
          f"resolution limit Kd ~ {T/2:.3g} nM\n")
    rows, items = [], []
    for name, reps in samples.items():
        try:
            res, d = analyse_sample(name, reps)
        except Exception as e:
            print(f"{name}: fit failed ({e})")
            continue
        rows.append(res)
        items.append((name, d, res))
        safe = "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in name)
        plot_sample(name, d, res, os.path.join(OUTPUT_DIR, safe))
    table = pd.DataFrame(rows)
    table.to_csv(os.path.join(OUTPUT_DIR, "results.csv"), index=False)
    if items:
        plot_grid(items, os.path.join(OUTPUT_DIR, "all_samples"))
    with open(os.path.join(OUTPUT_DIR, "methods.txt"), "w") as fh:
        fh.write(methods_text())

    show = ["sample", "n_repeats", "Kd_nM", "Kd_CI95_low", "Kd_CI95_high",
            "Kd_repeats_mean_nM", "Kd_repeats_SD_nM", "frac_bound_at_top", "flags"]
    with pd.option_context("display.max_colwidth", 80, "display.width", 200,
                           "display.float_format", "{:.3g}".format):
        print(table[[s for s in show if s in table.columns]].to_string(index=False))
    print("\n" + methods_text())
    shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
    return table

In [ ]:
# Upload the Excel/CSV file (Colab) or set PATH to a local file, then run.
PATH = None   # e.g. "my_plate.xlsx" when running locally

try:
    from google.colab import files
    up = files.upload()
    PATH = list(up.keys())[0]
except ImportError:
    pass

results = run(PATH)

try:
    files.download(OUTPUT_DIR + ".zip")
except NameError:
    print(f"Outputs in {OUTPUT_DIR}/ and {OUTPUT_DIR}.zip")
